In [ ]:

import mediapipe as mp
import numpy as np
import imutils
import cv2
import math

class handDetector():
    def __init__(self, mode=False, maxHands=2, detectionCon=0.5, trackCon=0.5):
        self.mode = mode
        self.maxHands = maxHands
        self.detectionCon = detectionCon
        self.trackCon = trackCon
        self.hands = mp.solutions.hands.Hands(static_image_mode=self.mode, max_num_hands=self.maxHands, min_detection_confidence=self.detectionCon, min_tracking_confidence=self.trackCon)
        self.mpDraw = mp.solutions.drawing_utils
        self.draw_style = self.mpDraw.DrawingSpec(thickness=2, circle_radius=2, color=(0,255,0))
        self.connect_style = self.mpDraw.DrawingSpec(thickness=1, color=(0,0,255))
        self.results = None

    def findHands(self, img, draw=True):
        h, w, c = img.shape
        x_min, y_min, x_max, y_max = 0, 0, 0, 0
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img.flags.writeable = False
        self.results = self.hands.process(imgRGB)
        img.flags.writeable = True

        if self.results.multi_hand_landmarks:
            for hlm in self.results.multi_hand_landmarks:
                x_list = [lm.x*w for lm in hlm.landmark]
                y_list = [lm.y*h for lm in hlm.landmark]
                x_min, x_max = max(0, int(min(x_list)-20)), min(w, int(max(x_list)+20))
                y_min, y_max = max(0, int(min(y_list)-20)), min(h, int(max(y_list)+20))

                if draw:
                    self.mpDraw.draw_landmarks(img, hlm, mp.solutions.hands.HAND_CONNECTIONS, self.draw_style, self.connect_style)
                    cv2.rectangle(img, (x_min,y_min), (x_max,y_max), (255,0,255), 2)

        return img, x_min, y_min, x_max, y_max

    def findPosition(self, img, handNo=0, draw=True):
        lmList = []
        bbox = []

        if self.results and self.results.multi_hand_landmarks:
            if handNo < len(self.results.multi_hand_landmarks):
                hlm = self.results.multi_hand_landmarks[handNo]
                h, w, c = img.shape

                for id, lm in enumerate(hlm.landmark):
                    cx, cy = int(lm.x*w), int(lm.y*h)
                    lmList.append([id, cx, cy])

                    if draw:
                        cv2.circle(img, (cx,cy), 5, (255,0,255), cv2.FILLED)

                x_list = [p[1] for p in lmList]
                y_list = [p[2] for p in lmList]
                bbox = [max(0,min(x_list)-20), max(0,min(y_list)-20), min(w,max(x_list)+20), min(h,max(y_list)+20)]

        return lmList, bbox

    def findDistance(self, p1, p2, lmList, img, draw=True):
        x1, y1 = lmList[p1][1], lmList[p1][2]
        x2, y2 = lmList[p2][1], lmList[p2][2]
        length = math.hypot(x2-x1, y2-y1)

        if draw:
            cv2.line(img, (x1,y1), (x2,y2), (255,0,255), 3)
            cv2.circle(img, (x1,y1), 8, (255,0,255), cv2.FILLED)
            cv2.circle(img, (x2,y2), 8, (255,0,255), cv2.FILLED)

        return length

In [16]:

detector = handDetector(maxHands=1)

cap = cv2.VideoCapture(1)
cap.set(3, 1280)
cap.set(4, 720)

while True:
    success, img = cap.read()
    if not success:
        print("Failed to capture frame")
        break

    img = cv2.flip(img, 1)
    img, x_min, y_min, x_max, y_max = detector.findHands(img)
    lmList, bbox = detector.findPosition(img)

    if len(lmList) != 0:
        length = detector.findDistance(4, 8, lmList, img)

        cv2.putText(img, f"Distance: {int(length)}", (30,30), cv2.FONT_HERSHEY_PLAIN, 2, (0,255,0), 2)
        
        if length > 100:
            cv2.putText(img, "Volume Up", (30,60), cv2.FONT_HERSHEY_PLAIN, 3, (0,255,0), 3)
        else:
            cv2.putText(img, "Volume Down", (30,60), cv2.FONT_HERSHEY_PLAIN, 3, (0,0,255), 3)


    cv2.imshow("Hand Volume Control", img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# full project

In [ ]:
import cv2
import mediapipe as mp
import math
import time

from pycaw.pycaw import AudioUtilities


class handDetector:
    def __init__(self, maxHands=1, detectionCon=0.5, trackCon=0.5):

        self.hands = mp.solutions.hands.Hands(
            max_num_hands=maxHands,
            min_detection_confidence=detectionCon,
            min_tracking_confidence=trackCon
        )

        self.mpDraw = mp.solutions.drawing_utils
        self.results = None

        self.draw_style = self.mpDraw.DrawingSpec(
            thickness=2,
            circle_radius=2,
            color=(0, 255, 0)
        )

        self.connect_style = self.mpDraw.DrawingSpec(
            thickness=1,
            color=(0, 0, 255)
        )

        self.volume = AudioUtilities.GetSpeakers().EndpointVolume


    def findHands(self, img, draw=True):

        h, w, _ = img.shape

        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        self.results = self.hands.process(rgb)

        box = [0, 0, 0, 0]

        if self.results.multi_hand_landmarks:

            for hand in self.results.multi_hand_landmarks:

                x = [int(lm.x * w) for lm in hand.landmark]
                y = [int(lm.y * h) for lm in hand.landmark]

                box = [
                    max(0, min(x) - 20),
                    max(0, min(y) - 20),
                    min(w, max(x) + 20),
                    min(h, max(y) + 20)
                ]

                if draw:

                    self.mpDraw.draw_landmarks(
                        img,
                        hand,
                        mp.solutions.hands.HAND_CONNECTIONS,
                        self.draw_style,
                        self.connect_style
                    )

                    cv2.rectangle(
                        img,
                        (box[0], box[1]),
                        (box[2], box[3]),
                        (255, 0, 255),
                        2
                    )

        return img, box


    def findPosition(self, img, handNo=0, draw=True):

        lmList = []

        if self.results and self.results.multi_hand_landmarks:

            if handNo < len(self.results.multi_hand_landmarks):

                hand = self.results.multi_hand_landmarks[handNo]

                h, w, _ = img.shape

                for id, lm in enumerate(hand.landmark):

                    cx = int(lm.x * w)
                    cy = int(lm.y * h)

                    lmList.append([id, cx, cy])

                    if draw:
                        cv2.circle(
                            img,
                            (cx, cy),
                            5,
                            (255, 0, 255),
                            cv2.FILLED
                        )

        return lmList


    def findDistance(self, p1, p2, lmList, img, draw=True):

        x1, y1 = lmList[p1][1:]
        x2, y2 = lmList[p2][1:]

        length = math.hypot(x2 - x1, y2 - y1)

        if draw:

            cv2.line(img, (x1, y1), (x2, y2), (255, 0, 255), 3)

            cv2.circle(img, (x1, y1), 8, (255, 0, 255), cv2.FILLED)
            cv2.circle(img, (x2, y2), 8, (255, 0, 255), cv2.FILLED)

        return length


    def volumeUp(self, amount=0.04):

        volume = self.volume.GetMasterVolumeLevelScalar()
        self.volume.SetMasterVolumeLevelScalar(
            min(1.0, volume + amount),
            None
        )


    def volumeDown(self, amount=0.04):

        volume = self.volume.GetMasterVolumeLevelScalar()
        self.volume.SetMasterVolumeLevelScalar(
            max(0.0, volume - amount),
            None
        )


# ---------------- MAIN ----------------

detector = handDetector(maxHands=1)

cap = cv2.VideoCapture(1)
cap.set(3,1200)
cap.set(4,720)
last_action = 0
cooldown = 0.5

while True:

    success, img = cap.read()

    if not success:
        break

    img = cv2.flip(img, 1)

    img, box = detector.findHands(img)
    lmList = detector.findPosition(img)

    if lmList:

        length = detector.findDistance(4, 8, lmList, img)

        now = time.time()

        if length > 100:

            action = "VOLUME UP"
            color = (0, 255, 0)

            if now - last_action > cooldown:
                detector.volumeUp()
                last_action = now

        elif length < 70:

            action = "VOLUME DOWN"
            color = (0, 0, 255)

            if now - last_action > cooldown:
                detector.volumeDown()
                last_action = now

        else:

            action = "NEUTRAL"
            color = (255, 255, 255)

        volume = detector.volume.GetMasterVolumeLevelScalar()

        cv2.putText(
            img, action, (30, 100),
            cv2.FONT_HERSHEY_PLAIN, 3, color, 3
        )

        cv2.putText(
            img, f"Distance: {int(length)}",
            (30, 150),
            cv2.FONT_HERSHEY_PLAIN, 2,
            (255, 255, 255), 2
        )

        cv2.putText(
            img, f"Volume: {int(volume * 100)}%",
            (30, 200),
            cv2.FONT_HERSHEY_PLAIN, 2,
            (255, 255, 255), 2
        )

    cv2.imshow("Hand Volume Control", img)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()
cv2.destroyAllWindows()